# OMOMO (human-object interaction) - Layer 1: rebuild the paper table from raw results

This notebook rebuilds every number of the paper's **OMOMO table** (`tables/exp_om.tex`, Tab. *OMOMO*) from the
per-sequence evaluation results stored in `release_data/omomo/L1_results/`. No GPU and no model are needed.

* **Our row (WMM)** is recomputed: each column is an average over the 482 test windows of a per-window value.
* **Baseline rows** (InterDiff, MDM, Pred-OMOMO, three CHOIS variants) are **quoted from the CHOIS paper**
  (Li et al., ECCV 2024, Table 1).

Set the environment variable `WMM_DATA` if your `release_data` folder lives somewhere else.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import omomo_l1 as L

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
L1 = L.l1_dir()
print('L1 results folder:', L1)
for p in sorted(L1.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(L1)}  ({p.stat().st_size/1024:.1f} KB)')

## 1. Our row: one record per test window

`metrics_raw.json` holds one record per test window (482 windows = the first 120-frame window of every
subject-16/17 sequence that has a text description, vacuum and mop excluded, as in CHOIS). Each record already
contains the per-window errors computed by the scorer from the predicted and ground-truth motion.

In [ ]:
ours_dir = L1 / L.OURS_DIR
per_window = L.load_per_window(ours_dir)
df = pd.DataFrame(per_window)
print(len(df), 'windows,', df.obj_name.nunique(), 'objects:', sorted(df.obj_name.unique()))
df[['stem', 'obj_name', 't_s_cm', 't_e_cm', 't_xy_cm', 'h_feet_cm', 'foot_sliding', 'cprec', 'crec', 'cf1',
    'gt_contact_frames', 'pred_contact_frames', 'n_frames_valid', 'mpjpe_mm', 'troot_mm', 't_obj_cm', 'o_obj']].head()

Averaging rules (identical to the original scorer):

| Column | per-window field | rule |
|---|---|---|
| T_s, T_e, T_xy, H_feet, T_obj, O_obj | `t_s_cm`, `t_e_cm`, `t_xy_cm`, `h_feet_cm`, `t_obj_cm`, `o_obj` | plain mean (already cm / unitless) |
| FS, MPJPE, T_root | `foot_sliding`, `mpjpe_mm`, `troot_mm` | mean, then mm -> cm (/10) |
| C_prec, C_rec | `cprec`, `crec` | plain mean |
| C_F1 | `cf1` | mean over windows whose ground truth has >= 1 contact frame (438 of 482) |
| C_% | `pred_contact_frames / n_frames_valid` | mean fraction of frames with predicted contact |

In [ ]:
ours = L.table_row(per_window)

# Cross-check against the aggregate the original scorer wrote next to the raw file.
stored = json.load(open(ours_dir / 'metrics.json'))
for c in L.COLUMNS:
    assert abs(ours[c] - stored[c]) < 1e-9, (c, ours[c], stored[c])
print('Recomputed row equals the scorer output metrics.json for all 13 columns.')
pd.DataFrame([ours], index=['WMM (recomputed)']).round(4)

## 2. The full table

In [ ]:
rows = {m: v for m, v in L.PAPER_TABLE.items() if m != 'WMM (ours)'}
table = pd.DataFrame(rows, index=L.COLUMNS).T
table.loc['WMM (ours, recomputed)'] = [float(L.round_half_up(ours[c], 2)) for c in L.COLUMNS]
table['source'] = ['quoted from CHOIS (ECCV 2024) Table 1'] * len(rows) + ['recomputed from L1_results']
print(table.to_string(na_rep='--'))

## 3. A look inside: per-object averages of our row

In [ ]:
df['MPJPE_cm'] = df.mpjpe_mm / 10
df['T_obj_cm'] = df.t_obj_cm
per_obj = df.groupby('obj_name')[['MPJPE_cm', 'T_obj_cm', 'cprec', 'crec']].mean().round(3)
per_obj['n_windows'] = df.groupby('obj_name').size()
print(per_obj.to_string())
ax = per_obj[['MPJPE_cm', 'T_obj_cm']].plot.bar(figsize=(9, 3.5))
ax.set_ylabel('cm (lower is better)')
ax.set_title('WMM on OMOMO: per-object mean error over the 482 test windows')
plt.tight_layout()
plt.show()